# Edge-Aware Pixel Art Converter

This program converts images, GIFs, and videos into pixel art using median filtering, dark-pixel averaging, and Canny edge detection.

### How to Use

1. Run the code cell below.
2. When prompted, upload an image, GIF, or video.
3. The algorithm will process the uploaded file using the default parameters.
4. The processed file will be downloaded automatically.

To adjust the pixelation effect, modify `BLOCK_SIZE` and the other parameters at the beginning of the code.

**Supported formats:** PNG, JPG, JPEG, GIF, MP4, AVI, MOV, MKV, and WEBM.


In [ ]:

import os
import subprocess

import cv2
import numpy as np

from PIL import Image, ImageSequence
from google.colab import files
from google.colab.patches import cv2_imshow


# --------------------------------------------------
# Algorithm Parameters
# --------------------------------------------------

BLOCK_SIZE = 10
DARK_THRESHOLD = 60
MIN_DARK_FRACTION = 0.30
MIN_EDGE_FRACTION = 0.05

CANNY_LOW_THRESHOLD = 50
CANNY_HIGH_THRESHOLD = 150


# --------------------------------------------------
# Divide an Image into Blocks
# --------------------------------------------------

def divide_into_blocks(image, block_size):
    """
    Divide an image into non-overlapping square blocks.

    Each block is flattened into a list of pixels.
    Images with dimensions not divisible by block_size
    are padded on the right and bottom edges.

    Returns:
        Color image: (block_rows, block_columns, pixels, channels)
        Grayscale image: (block_rows, block_columns, pixels)
    """

    image_height, image_width = image.shape[:2]

    # Calculate the padding needed for complete blocks.
    bottom_padding = (-image_height) % block_size
    right_padding = (-image_width) % block_size

    padding = [
        (0, bottom_padding),
        (0, right_padding)
    ]

    # Color images have an additional channel dimension.
    if image.ndim == 3:
        padding.append((0, 0))

    # Repeat the nearest border pixels when padding.
    padded_image = np.pad(image, padding, mode="edge")

    padded_height, padded_width = padded_image.shape[:2]

    number_of_block_rows = padded_height // block_size
    number_of_block_columns = padded_width // block_size

    pixels_per_block = block_size * block_size

    if image.ndim == 3:

        number_of_channels = image.shape[2]

        # Group pixels into square blocks.
        blocks = padded_image.reshape(
            number_of_block_rows,
            block_size,
            number_of_block_columns,
            block_size,
            number_of_channels
        )

        # Place the block row and column dimensions first.
        blocks = blocks.transpose(0, 2, 1, 3, 4)

        # Flatten each square block into a list of pixels.
        blocks = blocks.reshape(
            number_of_block_rows,
            number_of_block_columns,
            pixels_per_block,
            number_of_channels
        )

    else:

        blocks = padded_image.reshape(
            number_of_block_rows,
            block_size,
            number_of_block_columns,
            block_size
        )

        blocks = blocks.transpose(0, 2, 1, 3)

        blocks = blocks.reshape(
            number_of_block_rows,
            number_of_block_columns,
            pixels_per_block
        )

    return blocks


# --------------------------------------------------
# Edge-Aware Pixelation Algorithm
# --------------------------------------------------

def pixelate_image(
    image,
    block_size=BLOCK_SIZE,
    dark_threshold=DARK_THRESHOLD,
    min_dark_fraction=MIN_DARK_FRACTION,
    min_edge_fraction=MIN_EDGE_FRACTION
):
    """
    Convert an image into pixel art.

    General blocks use the median color.
    Blocks containing enough dark pixels and detected
    edges use the average color of their dark pixels.
    """

    image_height, image_width = image.shape[:2]

    # Step 1: Generate the edge map.

    # Calculate grayscale intensity by averaging channels.
    grayscale_image = image.mean(axis=2).astype(np.uint8)

    # Apply Gaussian smoothing before edge detection.
    blurred_image = cv2.GaussianBlur(
        grayscale_image,
        (3, 3),
        0
    )

    # Detect edges using the Canny algorithm.
    edge_map = cv2.Canny(
        blurred_image,
        CANNY_LOW_THRESHOLD,
        CANNY_HIGH_THRESHOLD
    )

    # Convert detected edges into Boolean values.
    edge_map = edge_map > 0

    # Divide the edge map into blocks.
    edge_blocks = divide_into_blocks(edge_map, block_size)

    # Calculate the fraction of edge pixels in each block.
    edge_fraction = edge_blocks.mean(axis=2)

    # Step 2: Divide the original image into blocks.

    image_blocks = divide_into_blocks(image, block_size)

    # Calculate the intensity of every pixel in each block.
    pixel_intensities = image_blocks.mean(axis=3)

    # Identify pixels below the dark intensity threshold.
    dark_pixel_mask = pixel_intensities < dark_threshold

    # Step 3: Calculate the median color of each block.

    median_colors = np.median(image_blocks, axis=2)

    # Step 4: Calculate the average of the dark pixels.

    # Count the dark pixels in each block.
    dark_pixel_count = dark_pixel_mask.sum(axis=2)

    # Expand the mask to apply it to all color channels.
    expanded_dark_mask = dark_pixel_mask[..., None]

    # Set non-dark pixels to zero before summation.
    dark_pixels = image_blocks * expanded_dark_mask

    # Sum the color values of qualifying dark pixels.
    dark_color_sum = dark_pixels.sum(axis=2)

    # Prevent division by zero when a block has no dark pixels.
    safe_dark_pixel_count = np.maximum(
        dark_pixel_count,
        1
    )

    dark_mean_colors = (
        dark_color_sum / safe_dark_pixel_count[..., None]
    )

    # Step 5: Determine which blocks preserve dark outlines.

    dark_fraction = dark_pixel_mask.mean(axis=2)

    has_enough_dark_pixels = (
        dark_fraction >= min_dark_fraction
    )

    has_enough_edge_pixels = (
        edge_fraction >= min_edge_fraction
    )

    # Both conditions must be satisfied.
    preserve_dark_outline = (
        has_enough_dark_pixels & has_enough_edge_pixels
    )

    # Step 6: Select the representative block colors.

    selected_colors = np.where(
        preserve_dark_outline[..., None],
        dark_mean_colors,
        median_colors
    )

    # Step 7: Reconstruct the image.

    # Expand each block color into a square of pixels.
    pixelated_image = np.repeat(
        selected_colors,
        block_size,
        axis=0
    )

    pixelated_image = np.repeat(
        pixelated_image,
        block_size,
        axis=1
    )

    # Remove padding and restore the original dimensions.
    pixelated_image = pixelated_image[
        :image_height,
        :image_width
    ]

    return pixelated_image.astype(np.uint8)


# --------------------------------------------------
# Process Static Images
# --------------------------------------------------

def process_image(file_path):

    image = cv2.imread(file_path)

    if image is None:
        raise ValueError("Unable to read the selected image.")

    pixelated_image = pixelate_image(image)

    output_path = "pixelated.png"

    cv2.imwrite(output_path, pixelated_image)

    cv2_imshow(pixelated_image)

    files.download(output_path)


# --------------------------------------------------
# Process Animated GIFs
# --------------------------------------------------

def process_gif(file_path):

    gif = Image.open(file_path)

    processed_frames = []
    frame_durations = []

    for frame in ImageSequence.Iterator(gif):

        # Convert each frame to an RGB NumPy array.
        rgb_frame = np.array(frame.convert("RGB"))

        # Apply the pixelation algorithm.
        pixelated_frame = pixelate_image(rgb_frame)

        # Convert the processed frame back to a PIL image.
        output_frame = Image.fromarray(pixelated_frame)

        processed_frames.append(output_frame)

        # Preserve the duration of each original frame.
        frame_duration = frame.info.get(
            "duration",
            gif.info.get("duration", 100)
        )

        frame_durations.append(frame_duration)

    output_path = "pixelated.gif"

    processed_frames[0].save(
        output_path,
        save_all=True,
        append_images=processed_frames[1:],
        duration=frame_durations,
        loop=0,
        disposal=2
    )

    files.download(output_path)


# --------------------------------------------------
# Process Videos
# --------------------------------------------------

def process_video(file_path):

    video_capture = cv2.VideoCapture(file_path)

    if not video_capture.isOpened():
        raise ValueError("Unable to open the selected video.")

    frame_rate = video_capture.get(cv2.CAP_PROP_FPS)

    frame_width = int(
        video_capture.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    frame_height = int(
        video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    if frame_rate <= 0:
        video_capture.release()
        raise ValueError("Unable to determine the video frame rate.")

    temporary_path = "raw.mp4"
    output_path = "pixelated.mp4"

    # Create a temporary video using the MP4V codec.
    video_codec = cv2.VideoWriter_fourcc(*"mp4v")

    video_writer = cv2.VideoWriter(
        temporary_path,
        video_codec,
        frame_rate,
        (frame_width, frame_height)
    )

    if not video_writer.isOpened():
        video_capture.release()
        raise ValueError("Unable to create the output video.")

    # Process each frame individually.
    while True:

        frame_available, frame = video_capture.read()

        if not frame_available:
            break

        pixelated_frame = pixelate_image(frame)

        video_writer.write(pixelated_frame)

    # Release video resources.
    video_capture.release()
    video_writer.release()

    # Convert the temporary video to H.264.
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel", "error",
            "-i", temporary_path,
            "-vcodec", "libx264",
            "-pix_fmt", "yuv420p",
            output_path
        ],
        check=True
    )

    files.download(output_path)


# --------------------------------------------------
# Upload and Process a File
# --------------------------------------------------

uploaded_files = files.upload()

if not uploaded_files:
    raise ValueError("No file was uploaded.")

input_file_path = next(iter(uploaded_files))

file_extension = os.path.splitext(
    input_file_path
)[1].lower()

video_extensions = (
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm"
)

if file_extension == ".gif":

    process_gif(input_file_path)

elif file_extension in video_extensions:

    process_video(input_file_path)

else:

    process_image(input_file_path)
